In [1]:
import numpy as np
import cmath
import matplotlib.pyplot as plt
import pde
from pde import DiffusionPDE, CartesianGrid, ScalarField, FieldCollection

In [ ]:
L = 10.0
k = 2.0
grid = CartesianGrid([(-L, L)], 128, periodic=[False])

x = grid.axes_coords[0]  # x-coordinates of the grid points
x0 = -6.0  # x0 < 0
sigma = 0.8

# Initial Condition
gaussian = np.exp(-((x - x0)**2) / (2*(sigma)**2))  #gaussian term 
plane_re = np.cos(k*x) * gaussian  #real part of the plane wave
plane_im = np.sin(k*x) * gaussian  #imaginary part of the plane wave
initial_cond_re = ScalarField(grid, plane_re)   #real part of the initial condition
initial_cond_im = ScalarField(grid, plane_im)   #imaginary part of the initial condition

state = FieldCollection([initial_cond_re, initial_cond_im], labels=["u", "v"])  #combine real and imaginary parts into a FieldCollection

In [3]:
# Boundary Conditions
k = 2.0
L = 10.0
hbar = 1.0  #scaled units for quantum PDE simulations
m = 1.0
omega = hbar * k**2 / (2 * m)
t = 10.0

# psi = lambda k,x : np.exp(1j * (k*x))  # plane wave solution, ref (1.1)

# old stuff
# # Real parts of Dirichlet and Robin
# bc_u = {"x-": {"value_expression": f"cos({k}*x - {omega}*t)"}, "x+": {"derivative": f"-{k} * v"}}  # Dirichlet at x=-L, Neumann at x=L

# # Imaginary parts of Dirichlet and Robin
# bc_v = {"x-": {"value_expression": f"sin({k}*x - {omega}*t)"}, "x+": {"derivative": f"{k} * u"}}  # Dirichlet at x=-L, Neumann at x=L

# new stuff
# BC operations combining parts of Dirichlet and Robin
bc = {
    "x-": {"derivative": 0},
    "x+": {"derivative": 0},
}

In [4]:
dx = x[1] - x[0]

def hook(state_data, t):
    u = state_data[0]
    v = state_data[1]

    # -------------------------
    # left boundary: Dirichlet
    # -------------------------
    theta = -k*L - omega*t
    u[0] = np.cos(theta)
    v[0] = np.sin(theta)

    # -------------------------
    # right boundary: coupled Robin
    # u_x = -k v
    # v_x =  k u
    # -------------------------
    c = dx * k
    denom = 1 + c**2

    u_in = u[-2]
    v_in = v[-2]

    u[-1] = (u_in - c*v_in) / denom
    v[-1] = (v_in + c*u_in) / denom

In [5]:
# Smoothed out Potential energy conditions for Interfaces

x_vals = np.linspace(-L, L, 1000)
a = 2.0  # width of the potential barrier
alpha = 10.0  # controls the steepness of the transition
v0 = 5.0  # height of the potential barrier

v_smooth = lambda x_vals, v0, a, alpha: 0.5 * v0 * (np.tanh(alpha * x_vals) - np.tanh(alpha * (x_vals-a))) 

In [6]:
# Governing equation / PDE

v_str = f"0.5 * {v0} * (tanh({alpha} * x) - tanh({alpha} * (x-{a})))"
# eq = pde.PDE({"psi": f"I * {hbar/(2*m)} * laplace(psi) - I / {hbar} * ({v_str}) * psi"})

eq = pde.PDE({
    "u": f"-( {hbar/(2*m)} ) * laplace(v) + ( ({v_str}) / {hbar} ) * v",
    "v": f"( {hbar/(2*m)} ) * laplace(u) - ( ({v_str}) / {hbar} ) * u"
}, bc = bc, post_step_hook=hook)

In [ ]:
# result = eq.solve(state, t_range=t)

result = eq.solve(
    state,
    t_range=t,
    dt=5e-5
)

  0%|          | 0/10.0 [00:00<?, ?it/s]

In [9]:
density0 = u0**2 + v0**2
norm = np.sqrt(np.trapz(density0, x))

u0 /= norm
v0 /= norm

u = result["u"].data
v = result["v"].data

density = u**2 + v**2
norm = np.sqrt(np.trapz(density, x))

u /= norm
v /= norm

density = u**2 + v**2

mask = x > a
T = np.trapz(density[mask], x[mask])

print("T =", T)
print("Total probability =", np.trapz(density, x))


NameError: name 'u0' is not defined